In [57]:
import pandas as pd
import subprocess

In [58]:
sample = "I04_1"

In [59]:
df_umap_coords = pd.read_csv(
    f"/gpfs/home/qcf21tku/scratch/refine_chap/{sample}_binarena_v3.pca.tsv", 
    sep = '\t',
    index_col = 0
)

df_umap_coords['contig_id'] = df_umap_coords.index

df_umap_coords['sample'] = df_umap_coords.contig_id.apply(lambda x: x.split('_')[0])
df_umap_coords = df_umap_coords.rename(columns = {"contig_id":"coa_contig_id"})
df_umap_coords.head()

,PC1,PC2,coa_contig_id,sample
k127_2619734,16.437687,3.096896,k127_2619734,k127
k127_2226775,-2.137753,1.161007,k127_2226775,k127
k127_1702831,-4.436522,1.282534,k127_1702831,k127
k127_1506352,7.464665,-0.674037,k127_1506352,k127
k127_65494,1.510566,-0.109252,k127_65494,k127


In [60]:

binning_dir = f"/gpfs/data/mock_lab/will/mosaic_tmp/binning_dir"
taxo_dir = f"/gpfs/data/mock_lab/will/mosaic_tmp/taxo_tmpdir"
mapping_folder = f"/gpfs/data/mock_lab/will/mosaic_tmp/coa_to_ma_mapping"
coassembly_folder = f"/gpfs/data/mock_lab/will/mosaic_tmp/coassemblies_tmp"

In [61]:
df_contigs = pd.read_csv(
    f"{mapping_folder}/{sample}.mapping.tsv", 
    sep = '\t', 
    names = ["coa_contig_id", "contig_id", "score"]
)

df_contigs['sample'] = df_contigs.contig_id.apply(lambda x: x.split("_")[0])

df_contigs['score_max'] = df_contigs.groupby('coa_contig_id')['score'].transform(max)

df_contigs = df_contigs[df_contigs.score == df_contigs.score_max]

df_contigs = df_contigs.loc[df_contigs[['coa_contig_id', 'score']].drop_duplicates().index]

df_contigs = df_contigs.drop(columns = ["score", "score_max"])

In [62]:
single_samples = list(set(df_contigs['sample']))

def make_taxo_df(sample_id):
    df = pd.read_csv(f"{taxo_dir}/{sample_id}/{sample_id}.faa.mms2lca", sep = '\t')
    df['sample'] = sample_id
    df['contig_id'] = df['sample'] + '_' + df['contig_id']
    return df

df_taxo = pd.concat(
    make_taxo_df(s) for s in single_samples
)

df = pd.merge(
    df_contigs, 
    df_taxo, 
    on = ['contig_id', 'sample'],
    how = 'left'
).fillna("Unknown")


favourite_groups = [
    "Bacillariophyta", 
    "Nucleocytoviricota", 
    "Mamiellophyceae",
    "Dinophyceae",
    "Metazoa",
    "Cryptophyceae",
    "Haptophyta", 
    "Chlorophyta", 
    "Euglenozoa",
    "Ciliophora",
    "Labyrinthulomycetes", 
    "Chrysophyceae", 
    "Choanoflagellata", 
    "Microbotryomycetes", 
    "Amoebozoa", 
    "Ochrophyta", 
    "Chytridiomycota"
]


def simplify_name(row):
    if row.superkingdom in ["Bacteria", "Archaea"]:
        return "Prokaryota"
    
    if row.superkingdom == "Unknown":
        if row['name'] == "Viruses":
            return "Viruses"
        return "Unknown"
    
    for x in favourite_groups:
        if x in row.lineage:
            return x

    if row.superkingdom == "Virus":
        return "Viruses"
    
    return row.superkingdom
    
    
df['name_simple'] = df.apply(simplify_name, axis = 1)

df['euk_unk'] = df.name_simple.isin(["Unknown", "Eukaryota"])

In [63]:
df = pd.merge(
    df_umap_coords, 
    df, 
    on = 'coa_contig_id', 
    how = 'left'
).fillna("None")

In [64]:
mbins = pd.DataFrame([x.split("\t") for x in subprocess.run(
    f"grep -H '>' {binning_dir}/{sample}/metabat_coa/*.fa | cut -f 10 -d '/' | sed 's|.fa:>|\t|g'", 
    capture_output = True, 
    shell = True
).stdout.decode("ASCII").strip().split("\n")], columns = ["metabat_id", "coa_contig_id"])

df = pd.merge(
    df, 
    mbins, 
    on = 'coa_contig_id', 
    how = 'left'
).fillna("None")

vbins = pd.DataFrame([x.split("\t") for x in subprocess.run(
    f"grep -H '>' {binning_dir}/{sample}/vamb_coa/*bins/*.fna | cut -f 1 -d ' ' | cut -f 11 -d '/' | sed 's|.fna:>|\t|g'", 
    capture_output = True, 
    shell = True
).stdout.decode("ASCII").strip().split("\n")], columns = ["vamb_id", "coa_contig_id"])

df = pd.merge(
    df, 
    vbins, 
    on = 'coa_contig_id', 
    how = 'left'
).fillna("None")

vacbins = pd.read_csv(
    f"{binning_dir}/{sample}/vamb_coa/vacuum_bins.tsv",
    sep = '\t'
).rename(
    columns = {'contig_id':'coa_contig_id'}
)

df = pd.merge(
    df, 
    vacbins, 
    on = 'coa_contig_id', 
    how = 'left'
).fillna("None")

In [65]:
df = pd.merge(
pd.merge(
    df, 
    pd.read_csv(
        f"/gpfs/home/qcf21tku/scratch/refine_chap/{sample}_binarena_v3.tsne.tsv", 
        sep = '\t',
        index_col = 0
    ), 
    left_on = 'coa_contig_id', 
    right_index = True
), 
    pd.read_csv(
        f"/gpfs/home/qcf21tku/scratch/refine_chap/{sample}_binarena_v3.umap.tsv", 
        sep = '\t',
        index_col = 0
    ), 
    left_on = 'coa_contig_id', 
    right_index = True
)

In [66]:
df = df[
    ~df.name_simple.isin(["Unknown", "Eukaryota", "None"])
]

df['size'] = '0.05'
df['opacity'] = '0.75'

df.loc[0, 'size'] = '1.0'
df.loc[0, 'opacity'] = '1.0'

In [67]:
df.to_csv(f"/gpfs/home/qcf21tku/scratch/refine_chap/{sample}_binarena_v3.all.tsv", sep = '\t')